# 06 · VAE condicional + selección de checkpoints con validación (DINOv2)

**Fase CRISP-DM:** 4. Modelado (entrenamiento VAE) y 5. Evaluación (solo selección, con **validación**).

**Parte A — Entrenamiento CVAE** (Kingma y Welling, 2014; Sohn et al., 2015): encoder/decoder convolucionales, latente 256, clase como *embedding* en encoder y decoder, pérdida MSE (suma por imagen) + β·KL con β = 1. 500 épocas sobre **train**, checkpoint cada 50 épocas. ≈ 15 min en L4.

**Parte B — Selección de checkpoints (reutilizable para GAN y difusión):**
- Imágenes reales de **validación a 128×128** (510; las mismas que verá cada generador a su resolución) → embeddings **DINOv2-ViT-L/14** (Stein et al., 2023), mismo preprocesamiento que el notebook 02 (224 bicúbico, normalización ImageNet, token CLS).
- Por cada checkpoint: se generan **1530 imágenes (90 por clase)**, mismo número y semilla para todos los checkpoints y técnicas.
- Métricas en el espacio DINOv2: **KD** (distancia de kernel, estimador insesgado de MMD² con kernel polinómico cúbico; Bińkowski et al., 2018), **FD** (distancia de Fréchet) y **precisión / recall k-NN** (Kynkäänniemi et al., 2019; k = 5).
- **Criterio de selección: menor KD**, porque es insesgado con muestras pequeñas (510 reales); FD se reporta como apoyo.
- **Test no se usa.**

**Entorno:** GPU **L4**, versión **2026.07**. Duración ≈ 30 min.

## Paso 1 · Entorno y parámetros

In [1]:
import sys, os, json, shutil, hashlib, zipfile, time, io, random, copy
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import scipy.linalg
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "⚠️ SIN GPU")

SEED = 42
RES = 128
EPOCHS = int(os.environ.get("EPOCHS", 500))
CKPT_EVERY = int(os.environ.get("CKPT_EVERY", 50))
BATCH_VAE = int(os.environ.get("BATCH_VAE", 64))
LATENTE = 256
N_GEN_POR_CLASE = int(os.environ.get("N_GEN_POR_CLASE", 90))     # 90 × 17 = 1530 por checkpoint
K_PR = 5
FEATURES = os.environ.get("FEATURES", "dinov2")                  # 'mock' solo para pruebas sin GPU
AMP = dict(device_type=DEVICE.type, dtype=torch.bfloat16)

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | GPU: NVIDIA L4


## Paso 2 · Drive y datasets (train y val a 128, verificados)

In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()
LOGS = PROJECT_DIR / "logs"
RUN_DIR = PROJECT_DIR / "checkpoints" / "cvae_cond128_seed42"
SEL_DIR = PROJECT_DIR / "seleccion"
for d in [RUN_DIR / "ckpt", RUN_DIR / "muestras", SEL_DIR, LOCAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG03 = json.loads((LOGS / "03_piloto.json").read_text())
from PIL import Image

def cargar_zip(nombre, sha):
    z = LOCAL_DIR / nombre
    if not z.exists():
        shutil.copy2(PROJECT_DIR / "datasets" / nombre, z)
    assert sha256_file(z) == sha, f"⚠️ {nombre} no coincide con el del notebook 03"
    with zipfile.ZipFile(z) as zf:
        lab = dict(json.loads(zf.read("dataset.json"))["labels"])
        nombres = sorted(lab)
        X = np.stack([np.asarray(Image.open(io.BytesIO(zf.read(n))).convert("RGB")) for n in nombres])
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.tensor([lab[n] for n in nombres])

X_train, Y_train = cargar_zip(LOG03["datasets"]["train_zip"], LOG03["datasets"]["train_sha256"])
X_val, Y_val = cargar_zip(LOG03["datasets"]["val_zip"], LOG03["datasets"]["val_sha256"])
N_CLASSES = int(Y_train.max()) + 1
print(f"train {tuple(X_train.shape)} | val {tuple(X_val.shape)} | clases {N_CLASSES} ✔")

Mounted at /content/drive
train (3060, 3, 128, 128) | val (510, 3, 128, 128) | clases 17 ✔


# Parte A · Entrenamiento CVAE

In [3]:
def bloque_down(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 4, 2, 1), nn.GroupNorm(8, cout), nn.SiLU())
def bloque_up(cin, cout):
    return nn.Sequential(nn.ConvTranspose2d(cin, cout, 4, 2, 1), nn.GroupNorm(8, cout), nn.SiLU())

class CVAE(nn.Module):
    def __init__(self, n_clases, z=LATENTE, emb=64):
        super().__init__()
        self.z = z
        self.emb = nn.Embedding(n_clases, emb)
        self.enc = nn.Sequential(bloque_down(3, 64), bloque_down(64, 128), bloque_down(128, 256),
                                 bloque_down(256, 256), bloque_down(256, 512))
        self.mu = nn.Linear(512 * 16 + emb, z)
        self.logvar = nn.Linear(512 * 16 + emb, z)
        self.fc = nn.Linear(z + emb, 512 * 16)
        self.dec = nn.Sequential(bloque_up(512, 256), bloque_up(256, 256), bloque_up(256, 128),
                                 bloque_up(128, 64), nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
    def encode(self, x, y):
        h = torch.cat([self.enc(x).flatten(1), self.emb(y)], 1)
        return self.mu(h), self.logvar(h)
    def decode(self, z, y):
        return self.dec(self.fc(torch.cat([z, self.emb(y)], 1)).view(-1, 512, 4, 4))
    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decode(z, y), mu, logvar

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
vae = CVAE(N_CLASSES).to(DEVICE)
opt = torch.optim.AdamW(vae.parameters(), lr=2e-4)
Xtr_dev, Ytr_dev = X_train.to(DEVICE), Y_train.to(DEVICE)
GEN = torch.Generator(device=DEVICE).manual_seed(SEED)
PASOS_EPOCA = len(Y_train) // BATCH_VAE

ckpts_previos = sorted((RUN_DIR / "ckpt").glob("cvae_ep*.pt"))
if ckpts_previos and f"cvae_ep{EPOCHS:04d}.pt" in [c.name for c in ckpts_previos]:
    print("✔ El entrenamiento ya estaba completo; se pasa a la selección.")
else:
    if ckpts_previos:
        ck = torch.load(ckpts_previos[-1], map_location=DEVICE, weights_only=False)
        vae.load_state_dict(ck["modelo"]); opt.load_state_dict(ck["opt"]); ep0 = ck["epoca"]
        GEN.set_state(ck["gen"].cpu())
        print("Se reanuda desde la época", ep0)
    else:
        ep0 = 0
    t0 = time.time()
    for ep in range(ep0 + 1, EPOCHS + 1):
        vae.train(); perd = []
        for _ in range(PASOS_EPOCA):
            idx = torch.randint(0, len(Ytr_dev), (BATCH_VAE,), device=DEVICE, generator=GEN)
            x = Xtr_dev[idx].float() / 127.5 - 1.0; y = Ytr_dev[idx]
            with torch.autocast(**AMP):
                xr, mu, logvar = vae(x, y)
            rec = F.mse_loss(xr.float(), x, reduction="sum") / BATCH_VAE
            kl = (-0.5 * (1 + logvar.float() - mu.float() ** 2 - logvar.float().exp()).sum()) / BATCH_VAE
            loss = rec + kl
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            perd.append((rec.item(), kl.item()))
        if ep % CKPT_EVERY == 0 or ep == EPOCHS:
            r, k = np.mean(perd, 0)
            torch.save({"modelo": vae.state_dict(), "opt": opt.state_dict(), "epoca": ep, "gen": GEN.get_state()},
                       RUN_DIR / "ckpt" / f"cvae_ep{ep:04d}.pt")
            print(f"época {ep:4d} | rec {r:8.1f} | KL {k:6.1f} | {(time.time()-t0)/60:.1f} min", flush=True)
    print(f"Entrenamiento VAE: {(time.time()-t0)/60:.1f} min")

Se reanuda desde la época 300
época  350 | rec    787.6 | KL  316.9 | 1.5 min
época  400 | rec    766.4 | KL  317.7 | 2.9 min
época  450 | rec    722.4 | KL  312.7 | 4.4 min
época  500 | rec    716.2 | KL  313.0 | 5.8 min
Entrenamiento VAE: 5.8 min


# Parte B · Selección de checkpoints con validación

## Paso B1 · Extractor DINOv2-ViT-L/14 y embeddings de validación

In [4]:
import torchvision.transforms.functional as TF
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

if FEATURES == "dinov2":
    dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14").eval().to(DEVICE)
else:                                   # solo para probar el flujo sin GPU ni descarga
    torch.manual_seed(0)
    dino = nn.Sequential(nn.AdaptiveAvgPool2d(8), nn.Flatten(), nn.Linear(3 * 64, 64)).eval().to(DEVICE)

@torch.no_grad()
def embeddings(x_m11, bs=128):
    # x en [-1, 1], (N,3,128,128) → (N, D) en float64
    out = []
    for i in range(0, len(x_m11), bs):
        x = ((x_m11[i:i + bs].float().cpu() + 1) / 2).clamp(0, 1)
        x = TF.resize(x, [224, 224], interpolation=TF.InterpolationMode.BICUBIC, antialias=True)
        x = ((x - MEAN) / STD).to(DEVICE)
        out.append(dino(x).float().cpu())
    return torch.cat(out).double().numpy()

F_VAL = embeddings(X_val.float() / 127.5 - 1.0)
print("Embeddings de validación:", F_VAL.shape)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:03<00:00, 325MB/s]


Embeddings de validación: (510, 1024)


## Paso B2 · Métricas (KD, FD, precisión y recall k-NN)

In [5]:
def fd(a, b):
    mu1, mu2 = a.mean(0), b.mean(0)
    s1, s2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    covmean = scipy.linalg.sqrtm(s1 @ s2)
    covmean = covmean[0] if isinstance(covmean, tuple) else covmean      # compatible con varias versiones de SciPy
    if not np.isfinite(covmean).all():                                    # estabilización estándar (pytorch-fid)
        off = np.eye(len(s1)) * 1e-6
        covmean = scipy.linalg.sqrtm((s1 + off) @ (s2 + off))
        covmean = covmean[0] if isinstance(covmean, tuple) else covmean
    covmean = covmean.real
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1 + s2 - 2 * covmean))

def kd(a, b):
    # MMD² insesgado con kernel polinómico (x·y/d + 1)^3, como en KID
    d = a.shape[1]
    k = lambda x, y: (x @ y.T / d + 1) ** 3
    kaa, kbb, kab = k(a, a), k(b, b), k(a, b)
    m, n = len(a), len(b)
    return float((kaa.sum() - np.trace(kaa)) / (m * (m - 1)) + (kbb.sum() - np.trace(kbb)) / (n * (n - 1))
                 - 2 * kab.mean())

def _dist(a, b):
    return np.sqrt(np.clip((a ** 2).sum(1)[:, None] + (b ** 2).sum(1)[None, :] - 2 * a @ b.T, 0, None))

def precision_recall(real, fake, k=K_PR):
    r_rad = np.sort(_dist(real, real), 1)[:, k]          # índice k: excluye la distancia a sí mismo (0)
    f_rad = np.sort(_dist(fake, fake), 1)[:, k]
    d_fr = _dist(fake, real)
    precision = float((d_fr <= r_rad[None, :]).any(1).mean())
    recall = float((d_fr.T <= f_rad[None, :]).any(1).mean())
    return precision, recall

def evaluar(f_fake):
    p, r = precision_recall(F_VAL, f_fake)
    return {"KD": kd(F_VAL, f_fake), "FD": fd(F_VAL, f_fake), "precision": p, "recall": r}

# Referencia real: mitad de val contra la otra mitad (escala de "lo mejor posible" con estas muestras)
rng = np.random.default_rng(SEED); perm = rng.permutation(len(F_VAL)); h = len(perm) // 2
REF = {"KD_val_vs_val": kd(F_VAL[perm[:h]], F_VAL[perm[h:]]), "FD_val_vs_val": fd(F_VAL[perm[:h]], F_VAL[perm[h:]])}
print("Referencia real (val/2 vs val/2):", {k: round(v, 4) for k, v in REF.items()})

Referencia real (val/2 vs val/2): {'KD_val_vs_val': -0.0086, 'FD_val_vs_val': 371.5414}


## Paso B3 · Selección para la VAE
Etiquetas fijas: 90 por clase. Ruido latente fijo (misma semilla) en todos los checkpoints.

In [6]:
import pandas as pd, torchvision.utils as vutils
Y_GEN = torch.arange(N_CLASSES).repeat_interleave(N_GEN_POR_CLASE)

@torch.no_grad()
def generar_vae(modelo, y, bs=256, seed=SEED):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    out = []
    for i in range(0, len(y), bs):
        yy = y[i:i + bs].to(DEVICE)
        z = torch.randn(len(yy), modelo.z, device=DEVICE, generator=g)
        out.append(modelo.decode(z, yy).float().cpu())
    return torch.cat(out).clamp(-1, 1)

filas = []
for ck_path in sorted((RUN_DIR / "ckpt").glob("cvae_ep*.pt")):
    ck = torch.load(ck_path, map_location=DEVICE, weights_only=False)
    m = CVAE(N_CLASSES).to(DEVICE); m.load_state_dict(ck["modelo"]); m.eval()
    x = generar_vae(m, Y_GEN)
    res = evaluar(embeddings(x))
    filas.append({"checkpoint": ck_path.name, "epoca": ck["epoca"], **res})
    print(f"{ck_path.name}: KD {res['KD']:.4f} | FD {res['FD']:.1f} | P {res['precision']:.3f} | R {res['recall']:.3f}", flush=True)

SEL_VAE = pd.DataFrame(filas).sort_values("epoca").reset_index(drop=True)
mejor = SEL_VAE.loc[SEL_VAE["KD"].idxmin()]
SEL_VAE.to_csv(SEL_DIR / "seleccion_vae.csv", index=False)
print("\n★ Checkpoint elegido (menor KD):", mejor["checkpoint"])
SEL_VAE

cvae_ep0050.pt: KD 15.8118 | FD 3224.9 | P 0.000 | R 0.000
cvae_ep0100.pt: KD 14.4714 | FD 3177.1 | P 0.000 | R 0.000
cvae_ep0150.pt: KD 13.8492 | FD 3119.1 | P 0.000 | R 0.000
cvae_ep0200.pt: KD 13.6478 | FD 3091.3 | P 0.000 | R 0.000
cvae_ep0250.pt: KD 13.6302 | FD 3082.7 | P 0.000 | R 0.000
cvae_ep0300.pt: KD 13.4495 | FD 3065.6 | P 0.000 | R 0.000
cvae_ep0350.pt: KD 13.3328 | FD 3052.2 | P 0.000 | R 0.000
cvae_ep0400.pt: KD 13.0719 | FD 3033.5 | P 0.000 | R 0.000
cvae_ep0450.pt: KD 12.9167 | FD 3021.3 | P 0.000 | R 0.000
cvae_ep0500.pt: KD 13.0192 | FD 3027.1 | P 0.000 | R 0.000

★ Checkpoint elegido (menor KD): cvae_ep0450.pt


,checkpoint,epoca,KD,FD,precision,recall
0,cvae_ep0050.pt,50,15.811845,3224.928244,0.0,0.0
1,cvae_ep0100.pt,100,14.471409,3177.072428,0.0,0.0
2,cvae_ep0150.pt,150,13.849221,3119.137489,0.0,0.0
3,cvae_ep0200.pt,200,13.647809,3091.258717,0.0,0.0
4,cvae_ep0250.pt,250,13.630194,3082.707095,0.0,0.0
5,cvae_ep0300.pt,300,13.449512,3065.618499,0.0,0.0
6,cvae_ep0350.pt,350,13.332817,3052.238818,0.0,0.0
7,cvae_ep0400.pt,400,13.071885,3033.479438,0.0,0.0
8,cvae_ep0450.pt,450,12.916683,3021.343043,0.0,0.0
9,cvae_ep0500.pt,500,13.019180,3027.135885,0.0,0.0


## Paso B4 · Gráfico de la selección y muestras del checkpoint elegido

In [7]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(SEL_VAE["epoca"], SEL_VAE["KD"], "o-"); ax[0].axhline(REF["KD_val_vs_val"], ls="--", c="gray", label="real vs real")
ax[0].set_title("KD (DINOv2) en validación ↓"); ax[0].legend()
ax[1].plot(SEL_VAE["epoca"], SEL_VAE["FD"], "o-"); ax[1].axhline(REF["FD_val_vs_val"], ls="--", c="gray")
ax[1].set_title("FD (DINOv2) en validación ↓")
ax[2].plot(SEL_VAE["epoca"], SEL_VAE["precision"], "o-", label="precisión")
ax[2].plot(SEL_VAE["epoca"], SEL_VAE["recall"], "s-", label="recall"); ax[2].set_ylim(0, 1); ax[2].legend()
ax[2].set_title("Precisión / recall k-NN")
for a in ax:
    a.axvline(mejor["epoca"], c="red", alpha=.4); a.set_xlabel("época")
fig.tight_layout(); fig.savefig(SEL_DIR / "seleccion_vae.png", dpi=150); plt.show()

ck = torch.load(RUN_DIR / "ckpt" / mejor["checkpoint"], map_location=DEVICE, weights_only=False)
m = CVAE(N_CLASSES).to(DEVICE); m.load_state_dict(ck["modelo"]); m.eval()
x = generar_vae(m, torch.arange(N_CLASSES).repeat(6), seed=123)
vutils.save_image((x + 1) / 2, SEL_DIR / "muestras_vae_elegido.png", nrow=N_CLASSES)
if IN_COLAB:
    from IPython.display import Image as IPImage, display
    display(IPImage(filename=str(SEL_DIR / "muestras_vae_elegido.png"), width=1000))

Output hidden; open in https://colab.research.google.com to view.

## Paso B5 · Registro

In [8]:
reg = {"fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
       "tecnica": "CVAE (MSE suma + KL, beta=1, latente 256)", "epocas": EPOCHS,
       "seleccion": {"criterio": "min KD DINOv2-ViT-L/14 en validacion 128px",
                     "n_generadas": int(len(Y_GEN)), "n_val": int(len(F_VAL)), "k_pr": K_PR,
                     "elegido": mejor["checkpoint"], "metricas_elegido": {k: float(mejor[k]) for k in ["KD", "FD", "precision", "recall"]},
                     "referencia_real": REF},
       "features": FEATURES,
       "versiones": {"python": sys.version.split()[0], "torch": torch.__version__}}
(LOGS / "06_vae_seleccion.json").write_text(json.dumps(reg, indent=2, ensure_ascii=False))
print(json.dumps(reg["seleccion"], indent=2, ensure_ascii=False))

{
  "criterio": "min KD DINOv2-ViT-L/14 en validacion 128px",
  "n_generadas": 1530,
  "n_val": 510,
  "k_pr": 5,
  "elegido": "cvae_ep0450.pt",
  "metricas_elegido": {
    "KD": 12.91668305955741,
    "FD": 3021.34304343116,
    "precision": 0.0,
    "recall": 0.0
  },
  "referencia_real": {
    "KD_val_vs_val": -0.00856911470014765,
    "FD_val_vs_val": 371.54143101758433
  }
}


**Al terminar:** desconecta la GPU. Envía al asesor la tabla del Paso B3, el gráfico y la malla de muestras.